# 音楽音源分離の評価指標

音源分離手法の良し悪しを測るには、**客観的指標**と**主観評価**の２つの手段があり、それぞれ一長一短があります。客観的指標は分離結果から直ぐに計算でき、数値的な比較もできますが、どの指標も人間の感覚とギャップがあるので、音源分離の品質を厳密に反映することはできません。分離品質の良し悪しを判定する究極の基準は主観評価、つまり直接聞き比べるしかありませんが、やはり主観評価はコストの手間もかかります。必要に応じて使い分けましょう。

## 標準的な評価指標と評価ツール

音源分離問題で最も一般的な評価指標[^VGFevotte06]では、分離された信号$\hat{\boldsymbol{s}}_i$は以下の成分からなると仮定されています。

$$
\hat{\boldsymbol{s}}_i=\boldsymbol{s}_{target}+\boldsymbol{e}_{interf}+\boldsymbol{e}_{noise}+\boldsymbol{e}_{artif}
$$

[^VGFevotte06]: Emmanuel Vincent, Rémi Gribonval, and Cédric Févotte. Performance measurement in blind audio source separation. IEEE transactions on audio, speech, and language processing, 14(4):1462–1469, 2006. url: https://ieeexplore.ieee.org/document/1643671

$\boldsymbol{s}_{target}$は真の音源、$\boldsymbol{e}_{interf}$は別音源の漏れ込み、$\boldsymbol{e}_{noise}$はノイズ成分、$\boldsymbol{e}_{artif}$はアーティファクト（不自然な音？）成分を指します。正解データがあれば、$\boldsymbol{s}_{target}$は既知の信号ですが、$\boldsymbol{e}_{***}$の成分は未知なので、いくつかの仮定を置いて推定されます。

音源分離タスクの評価指標は、以上の成分に基づいて以下のように定義されます。いずれも単位はdBで、高ければ高いほど良い数値です。

**SDR(Source to Distortion Ratio)**

$$
SDR =10\log_{10}\frac{\sum_n\Vert\boldsymbol{s}_{target}\Vert^2}{\sum_n\Vert\boldsymbol{e}_{interf}+\boldsymbol{e}_{noise}+\boldsymbol{e}_{artif}\Vert^2}
$$

$\boldsymbol{e}_{interf}+\boldsymbol{e}_{noise}+\boldsymbol{e}_{artif}=\hat{\boldsymbol{s}}-\boldsymbol{s}_{target}$なので、未知成分を推定する必要がなく比較的簡単に計算できる指標です。分離音の全体的な品質を測る、音源分離の最も主要な指標で、SDRのみを報告する論文も多いです。

ボーカル分離のSDRのベースラインがだいたい6-7dBで、SOTAは10dBを上回る水準に達しています。


**SAR(Source to Artifacts Ratio)**

$$
SAR=10\log_{10}\frac{\sum_n\Vert\boldsymbol{s}_{target}+\boldsymbol{e}_{interf}+\boldsymbol{e}_{noise}\Vert^2}{\sum_n\Vert\boldsymbol{e}_{artif}\Vert^2}
$$

分離音の中にアーティファクトが含まれていないか測る指標です。分離手法が変な歪みを導入していないか確かめるためのものです。

**SIR(Source to Interference Ratio)**

$$
SIR=10\log_{10}\frac{\sum_n\Vert\boldsymbol{s}_{target}\Vert^2}{\sum_n\Vert\boldsymbol{e}_{interf}\Vert^2}
$$

別音源からの漏れ込み(bleeding)を抑制できているか測る指標です。


### 評価ツール
評価ツールは以下のライブラリが存在します。

* `museval`: 音源分離モデルOpenUnmixや、標準的な音楽音源分離データセットMUSDB18を提供しているコミュニティSigSepが提供している評価ライブラリ (https://github.com/sigsep/sigsep-mus-eval)。Music Demixing Challengeなどの音楽音源分離コンテストでも採用されており、評価ツールのデファクトスタンダードと見なされています。**SDR、SAR、SIR**指標のほかに、**ISR** (**Image to Spatial Ratio**)という指標も計算します。

* `torchmetrics`: PyTorch Lightningコミュニティが開発している評価ライブラリ (https://github.com/Lightning-AI/torchmetrics)。分類・クラスタリング・検索などの基本的タスクから、マルチモーダルやLLM関連の最先端トピックまで、幅広い機械学習分野の評価指標が実装されているライブラリです。SDRをはじめ音に関するタスクの評価指標が多数実装されています。`museval`のようにフレームごとに計算するのではなく、入力信号全体に対して指標を計算する関数なので、`museval`の計算結果とは一致しません。

In [21]:
import librosa
import museval
import numpy as np
from IPython.display import Audio, display

y_mixture, sr = librosa.load("assets/mixture.flac", sr=None, mono=True)

y_ref_vocal, sr = librosa.load("assets/orig_vocals.flac", sr=None, mono=True)
y_ref_drums, sr = librosa.load("assets/orig_drums.flac", sr=None, mono=True)
y_ref_bass, sr = librosa.load("assets/orig_bass.flac", sr=None, mono=True)
y_ref_other, sr = librosa.load("assets/orig_other.flac", sr=None, mono=True)

y_est_vocal, sr = librosa.load("assets/estimated_vocals.flac", sr=None, mono=True)
y_est_drums, sr = librosa.load("assets/estimated_drums.flac", sr=None, mono=True)
y_est_bass, sr = librosa.load("assets/estimated_bass.flac", sr=None, mono=True)
y_est_other, sr = librosa.load("assets/estimated_other.flac", sr=None, mono=True)

print("ミックス音源")
display(Audio(y_mixture, rate=sr))
print("オリジナルのボーカル音源")
display(Audio(y_ref_vocal, rate=sr))
print("HDemucsによる推定ボーカル音源")
display(Audio(y_est_vocal, rate=sr))    


ミックス音源


オリジナルのボーカル音源


HDemucsによる推定ボーカル音源


In [22]:
y_ref = np.stack([
    y_ref_vocal,
    y_ref_drums,
    y_ref_bass,
    y_ref_other
], axis=0)[... , None]
y_est = np.stack([
    y_est_vocal,
    y_est_drums,
    y_est_bass,
    y_est_other
], axis=0)[... , None]


scores = museval.metrics.bss_eval(
    reference_sources = y_ref,
    estimated_sources= y_est,
)

sdr = scores[0][0].mean()
isr = scores[1][0].mean()
sir = scores[2][0].mean()
sar = scores[3][0].mean()

print("ボーカル分離評価指標（museval）:")
print(f"SDR: {sdr:.2f} dB")
print(f"ISR: {isr:.2f} dB")
print(f"SIR: {sir:.2f} dB")
print(f"SAR: {sar:.2f} dB")

ボーカル分離評価指標（museval）:
SDR: 11.51 dB
ISR: 21.97 dB
SIR: 21.81 dB
SAR: 11.87 dB


In [23]:
import torchmetrics
import torch

y_ref_torch = torch.from_numpy(y_ref.squeeze()).float()
y_est_torch = torch.from_numpy(y_est.squeeze()*5.0).float()
sdr_tm = torchmetrics.functional.audio.signal_distortion_ratio(y_ref_torch.squeeze(), y_est_torch.squeeze())
sdr_tm_vocal = sdr_tm[0].item()

print("ボーカル分離評価指標（torchmetrics）:")
print(f"SDR: {sdr_tm_vocal:.2f} dB")


ボーカル分離評価指標（torchmetrics）:
SDR: 14.40 dB


## 主観評価指標：MOS

主観評価で広く使われている指標の一つが**MOS** (**Mean Opinion Score**)です。評価者に各サンプルの品質を1~5の範囲で点数を付けてもらい、その評価値の平均を評価指標とします。

音源分離モデルDemucsの論文[^Demucs19]では、提案手法とベースラインを含む3つの音源分離手法を比較するために、以下の手順で主観評価を実施しました。
* テストセット上の分離結果から長さ8秒のサンプルをランダムに切り出す。
* 38名の評価者に、それぞれ20サンプルをランダムに提示し、分離品質を点数付けしてもらう。サンプルは３つの手法の分離結果および正解データからランダムに選ばれ、評価者はどの手法によるものかは知らない。
* MOSを集計する。

[^Demucs19]: Alexandre D´ efossez, Nicolas Usunier, L´ eon Bottou, and Francis Bach, “Music source separation in the waveform domain,” 2019. url: https://arxiv.org/abs/1911.13254


より正確な考察を得るために、注目点を雑音(artifacts)と漏れ込み(bleeding)に分けて、２通りのMOSを測ることもできます[^HDemucs21]。

![image](assets/MOS.png)

[^HDemucs21]: Alexandre D´efossez, “Hybrid spectrogram and waveform source separation,” in Proceedings of the ISMIR 2021 Workshop on Music Source Separation, 2021. url: https://arxiv.org/abs/2111.03600